# YOLO26 fine-tune on Colab — CVAT zip 업로드

Phase 1: 정지 차량 감지용 YOLO26(car 단일 클래스) fine-tune.

**라벨링: CVAT**. CVAT에서 bbox 라벨링 후 **YOLO 포맷으로 export → zip**을 받아
이 노트북에 업로드한다 (Drive 불필요).

사용 순서:
1. **런타임 → 런타임 유형 변경 → GPU (T4/L4)**
2. CVAT에서 car bbox 라벨링 → **Export → `YOLO 1.1`** (또는 `Ultralytics YOLO`) → zip 다운로드
3. 아래 **업로드 셀**에서 그 zip 업로드 (`data.yaml` + `images/` + `labels/` 구조)
4. `CONFIG` 채우고 위에서부터 실행
5. 끝나면 `best.pt`를 다운로드해서 `extract_labels.py --yolo_weights <best.pt>` 로 사용

> CVAT `YOLO 1.1` export는 `obj.data` / `obj.names` / `obj_train_data/` 구조,
> `Ultralytics YOLO` export는 `data.yaml` + `images/`+`labels/` 구조다. 아래 셀은
> 둘 다 처리하도록 `data.yaml`을 찾거나 없으면 생성한다.

## 1. 설치

In [ ]:
!pip install -q -U ultralytics   # YOLO26 지원 위해 최신 ultralytics
import ultralytics, torch
print('ultralytics', ultralytics.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 2. CONFIG — 여기만 손대면 됨

CVAT export zip을 업로드(다음 섹션)하고, 학습 하이퍼파라미터만 여기서 조정.
Jetson 배포 입력 크기와 맞추려고 `IMGSZ=320`.

In [ ]:
MODEL    = 'yolo26n.pt'   # YOLO26: n / s / m / l / x (NMS-free, edge 최적화)
EPOCHS   = 100
IMGSZ    = 320            # Jetson 배포 입력 크기와 맞춤
BATCH    = 32             # T4 16GB 기준. OOM이면 16
PATIENCE = 30             # early-stop
RUN_NAME = 'car_v1'
OUTPUT_DIR = '/content/yolo_runs'
DATA_ROOT  = '/content/cvat_yolo'   # 업로드 zip을 푸는 곳
DATA_DIR   = '/content/yolo_ds'     # ultralytics 표준 구조로 재배치할 곳
VAL_FRAC   = 0.2

## 3. CVAT zip 업로드 + 압축해제

CVAT에서 받은 YOLO export zip을 업로드한다. `data.yaml`이 있으면 그대로 쓰고,
없으면(`YOLO 1.1` 구조) `obj.names`로부터 생성한다.

In [ ]:
import os, glob, zipfile, shutil, random, yaml
from google.colab import files

# 1) zip 업로드. CVAT zip은 한글 파일명이 이미지=CP949 / txt=UTF-8로 섞여 저장돼
#    그냥 풀면 stem이 안 맞는다. 엔트리별로 원본 인코딩을 복원하며 직접 추출한다.
shutil.rmtree(DATA_ROOT, ignore_errors=True)
os.makedirs(DATA_ROOT, exist_ok=True)
up = files.upload()                     # CVAT YOLO 1.1 export zip 선택
zip_name = next(iter(up))

def recover_name(info):
    raw = info.filename
    if info.flag_bits & 0x800:           # 이미 UTF-8 플래그면 그대로
        return raw
    b = raw.encode('cp437', errors='replace')   # zipfile이 cp437로 디코드한 걸 되돌림
    for enc in ('cp949', 'euc-kr', 'utf-8'):
        try:
            return b.decode(enc)
        except Exception:
            continue
    return raw

with zipfile.ZipFile(zip_name) as z:
    for info in z.infolist():
        if info.is_dir():
            continue
        name = recover_name(info).replace('\\', '/')
        dst = os.path.join(DATA_ROOT, name)
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        with z.open(info) as src, open(dst, 'wb') as out:
            shutil.copyfileobj(src, out)
print('extracted (name-recovered) to', DATA_ROOT)

# 2) 클래스명: obj.names → data.yaml → 기본 ['car']
names = None
nf = glob.glob(os.path.join(DATA_ROOT, '**', 'obj.names'), recursive=True)
if nf:
    names = [l.strip() for l in open(nf[0]) if l.strip()]
else:
    yf = glob.glob(os.path.join(DATA_ROOT, '**', 'data.yaml'), recursive=True)
    if yf:
        names = yaml.safe_load(open(yf[0])).get('names')
if not names:
    names = ['car']
print('names:', names)

# 3) 이미지 ↔ txt를 파일명(stem)으로 짝짓기
IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp')
all_files = [p for p in glob.glob(os.path.join(DATA_ROOT, '**', '*'), recursive=True)
             if os.path.isfile(p)]
imgs = [p for p in all_files if os.path.splitext(p)[1].lower() in IMG_EXT]
META_TXT = {'classes.txt', 'obj.names', 'train.txt', 'val.txt', 'obj.data'}
txt_by_stem = {}
for p in all_files:
    if p.lower().endswith('.txt') and os.path.basename(p) not in META_TXT:
        txt_by_stem.setdefault(os.path.splitext(os.path.basename(p))[0], p)
print('images found:', len(imgs), '| label txt found:', len(txt_by_stem))

pairs = [(im, txt_by_stem.get(os.path.splitext(os.path.basename(im))[0])) for im in imgs]
with_lbl = [pr for pr in pairs if pr[1]]
print('images WITH matching label:', len(with_lbl), '/', len(pairs))
assert with_lbl, ('이미지와 stem이 같은 .txt를 못 찾음. zip 안 .txt 존재 / 파일명 일치 확인.')

# 4) ultralytics 표준 구조로 복사 + split
shutil.rmtree(DATA_DIR, ignore_errors=True)
for sub in ('images/train','images/val','labels/train','labels/val'):
    os.makedirs(os.path.join(DATA_DIR, sub), exist_ok=True)
random.Random(0).shuffle(pairs)
n_val = max(1, int(len(pairs) * VAL_FRAC))
for k, (im, txt) in enumerate(pairs):
    split = 'val' if k < n_val else 'train'
    base = os.path.basename(im); stem = os.path.splitext(base)[0]
    shutil.copy(im, os.path.join(DATA_DIR, 'images', split, base))
    if txt:
        shutil.copy(txt, os.path.join(DATA_DIR, 'labels', split, stem + '.txt'))

# 5) data.yaml
DATA_YAML = os.path.join(DATA_DIR, 'data.yaml')
with open(DATA_YAML, 'w') as f:
    yaml.safe_dump({'path': DATA_DIR, 'train': 'images/train', 'val': 'images/val',
                    'nc': len(names), 'names': names}, f, allow_unicode=True)
n_tr = len(glob.glob(os.path.join(DATA_DIR,'labels/train','*.txt')))
n_va = len(glob.glob(os.path.join(DATA_DIR,'labels/val','*.txt')))
print('DATA_YAML:', DATA_YAML)
print(f'train: {len(pairs)-n_val} imgs / {n_tr} labels | val: {n_val} imgs / {n_va} labels')
assert n_tr > 0, 'train 라벨 0개'

## 4. 학습

폐쇄 환경 + 단일 클래스라 mosaic는 약하게, flip은 끔(방향 정보 보존).
결과는 `OUTPUT_DIR/RUN_NAME/weights/best.pt`.

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL)
results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    project=OUTPUT_DIR,
    name=RUN_NAME,
    exist_ok=True,
    device=0,
    mosaic=0.5,
    mixup=0.0,
    fliplr=0.0,   # 좌우 flip 금지 (방향 정보)
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
)

## 5. 검증

In [ ]:
best_pt = os.path.join(OUTPUT_DIR, RUN_NAME, 'weights', 'best.pt')
print('best.pt:', best_pt, 'exists=', os.path.exists(best_pt))

best = YOLO(best_pt)
metrics = best.val(data=DATA_YAML, imgsz=IMGSZ, split='val')
print('mAP50   :', float(metrics.box.map50))
print('mAP50-95:', float(metrics.box.map))
print('names   :', best.names)

## 6. best.pt 다운로드

Drive 안 쓰므로 브라우저로 직접 다운로드. WSL에서 받아 `extract_labels.py --yolo_weights`로 사용.

In [ ]:
from google.colab import files
files.download(best_pt)

## 7. (선택) ONNX export

Jetson TRT engine용 중간 산출물. WSL standalone inference만이면 생략 가능.
TRT engine 빌드(trtexec)는 반드시 Jetson에서.

In [ ]:
best.export(format='onnx', imgsz=IMGSZ, opset=12, simplify=True)
onnx_path = best_pt.replace('.pt', '.onnx')
print('ONNX:', onnx_path, 'exists=', os.path.exists(onnx_path))
files.download(onnx_path)